<a href="https://colab.research.google.com/github/jayanthmcs24-sudo/AIML-lab/blob/main/1WN24CS122_Week7_FOL_Unification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
class Expression:

    def __init__(self, symbol, args=None, is_variable=False):
        self.symbol = symbol
        self.args = args or []
        self.is_var = is_variable

    def is_variable(self):
        return self.is_var and len(self.args) == 0

    def is_atom(self):

        return len(self.args) == 0

    def occurs_in(self, expr):
        if self == expr:
            return True
        if not expr.is_atom():
            return any(self.occurs_in(arg) for arg in expr.args)
        return False

    def apply_substitution(self, sub):
        if self in sub:
            return sub[self].apply_substitution(sub)
        if not self.is_atom():
            new_args = [arg.apply_substitution(sub) for arg in self.args]
            return Expression(self.symbol, new_args, self.is_var)
        return self

    def __eq__(self, other):
        if not isinstance(other, Expression):
            return False
        return (self.symbol == other.symbol and
                self.is_var == other.is_var and
                self.args == other.args)

    def __hash__(self):
        return hash((self.symbol, self.is_var, tuple(self.args)))

    def __repr__(self):
        if not self.args:
            return self.symbol
        args_str = ", ".join(map(str, self.args))
        return f"{self.symbol}({args_str})"


def unify(psi1, psi2):

    if psi1.is_atom() or psi2.is_atom():
        if psi1 == psi2:
            return {}
        elif psi1.is_variable():
            if psi1.occurs_in(psi2):
                return "FAILURE"

            else:
                return {psi1: psi2}
        elif psi2.is_variable():
            if psi2.occurs_in(psi1):
                return "FAILURE"
            else:
                return {psi2: psi1}

        else:
            return "FAILURE"

    if psi1.symbol != psi2.symbol:
        return "FAILURE"

    if len(psi1.args) != len(psi2.args):
        return "FAILURE"

    subst = {}  # NIL / empty dict

    l1 = list(psi1.args)
    l2 = list(psi2.args)

    for i in range(len(l1)):
        s = unify(l1[i], l2[i])

        if s == "FAILURE":
            return "FAILURE"

        if s != {}:
            l1 = [arg.apply_substitution(s) for arg in l1]
            l2 = [arg.apply_substitution(s) for arg in l2]

            for k, v in subst.items():
                subst[k] = v.apply_substitution(s)
            subst.update(s)

    return subst


if __name__ == "__main__":
    # Define variables and constants
    x = Expression("x", is_variable=True)
    y = Expression("y", is_variable=True)
    a = Expression("a", is_variable=False)
    b = Expression("b", is_variable=False)


    psi1 = Expression("P", [x, a])
    psi2 = Expression("Q", [x, a])


    result = unify(psi1, psi2)

    print(f"Psi1: {psi1}")
    print(f"Psi2: {psi2}")
    if isinstance(result, dict):
        sub_str = ", ".join([f"{val}/{var}" for var, val in result.items()])
        print(f"Result (SUBST): {{{sub_str}}}")
    else:
        print(f"Result: {result}")

Psi1: P(x, a)
Psi2: Q(x, a)
Result: FAILURE
